# Fine-Tuningwith Vertex AI

**Module 9 · Lesson 9.3**  
*Netsetos GenAI Engineering Course v2.0*

**In this lesson you will:**
- The Managed Pipeline + the Data Format
- Launch the Tuning Job - the Current SDK
- Hyperparameters vs Overfitting
- Cost & Quotas - and No Inference Premium
- Use & Evaluate the Tuned Model
- SFT / DPO / Continuous / Gemma - Managed vs Open

---

> This notebook is generated from the published lesson HTML. The HTML is the source of truth - do not hand-edit this notebook. Re-run the generator if the lesson updates.


In [ ]:
# Install Python dependencies used by this lesson (uncomment on Colab / fresh env).
# !pip install google-genai python-dotenv -q  # noqa


In [ ]:
import os
# Load API keys from the environment (or a .env file via python-dotenv).
# Never hardcode keys. Any ONE provider key is enough to start;
# the multi-provider demos are best with two or more.
os.environ.setdefault("GOOGLE_API_KEY", "")     # aistudio.google.com


## Theimport vertexaiThat Stopped Working

> 💡 **Info**
>
> One morning in June 2026, a team’s nightly fine-tuning pipeline failed with a single line: `ModuleNotFoundError: No module named 'vertexai.tuning'`. Nothing had changed in their code - but Google had done exactly what it announced a year earlier: on **2026-06-24** it **removed** the old `vertexai` Generative AI SDK. Every `vertexai.tuning.sft.train(...)` and `GenerativeModel(...)` call stopped importing. The fix was a 20-minute migration to the unified **`google-genai`** SDK - one `Client` object, a renamed config, same job. **That is the SDK this lesson uses from line one**, so you never write code that has an expiry date. (Same lesson, bigger picture: on a managed platform, currency is not optional - the deprecation calendar is part of your architecture.)

### 🎯 What you will be able to do after this lesson

- **Build** a Gemini tuning dataset in the right JSONL shape (`contents` / `role: model` / `parts`) and launch a supervised tuning job with the current `google-genai` SDK.

- **Operate** the job lifecycle: choose hyperparameters (Flash defaults vs Pro-small-data), poll job state, and call the auto-deployed tuned endpoint.

- **Compare** managed Gemini tuning vs open-weight Gemma, and the cost of each - including why a tuned model has no inference premium.

- **Evaluate** a tuned model (base-vs-tuned, the Gen AI Eval Service), read overfitting from the loss curves, and handle India regions + DPDP.

> 📦 **Info**
>
> ✅ Before you startThis is where the curated JSONL from **Lesson 9.2** goes to actually train a model - the managed, zero-GPU path. You should know the fine-tuning decision (9.1) and the data format (9.2). The open-weight / self-hosted alternative is **Lesson 9.4**; the LoRA/QLoRA mechanics under the hood are **9.5**.

## 60-Second Warm-Up

Flip each card and answer before revealing.

> 🎓 **Analogy**
>
> **Vertex AI tuning is hiring a personal tutor for Gemini.** Gemini already graduated university (pre-training); it just does not know YOUR company. The tutor (your data) teaches it your tone, products, and policies - no building a school, no buying desks, no hiring faculty. You upload data and pay by the token. *Where the analogy breaks down:* the tutor only ever works in Google’s classroom - the tuned model is a LoRA adapter served on Google’s shared endpoint, and you cannot take the student home. If you need the weights on your own machines, you hire a different tutor: **Gemma** (open weights).

> 📦 **Info**
>
> ⚠️ The misconception that sets wrong expectations**“Vertex ‘fine-tuning’ rewrites all of Gemini’s weights.”** It does not. Managed Gemini tuning is **LoRA adapter tuning**: the base model stays frozen and only small low-rank adapter matrices are trained. That is why it is fast (minutes to hours), tiny (megabytes, not gigabytes), keeps the base model’s general skills - and why you get **no downloadable weights** (the adapter runs only on Google’s endpoint). You are teaching behavior, not rebuilding the model.

> 💡 **Info**
>
> 🚫 Anti-pattern: default hyperparameters for Pro on a small datasetThe most common Vertex tuning mistake is accepting the defaults everywhere. For **Gemini Pro on a small dataset**, Google specifically recommends an aggressive `learning_rate_multiplier=10` and `epoch_count=20` - the defaults badly underfit there. (For Flash, defaults are fine.) Always pair it with a validation set + checkpoints so you can catch overfitting.

---

## 🎯 Concept 1: The Managed Pipeline + the Data Format

### The Managed Pipeline + the Data Format

Six stages, and the one format detail that fails jobs: role ‘model’, parts, not a string.

The whole managed pipeline is six stages: **prepare JSONL → upload to GCS → launch a job → train (Google’s GPUs) → evaluate → auto-deploy**. You touch only the ends: format the data and call the tuned endpoint. The one detail that trips everyone is the format - Gemini does *not* use OpenAI’s shape. It wants `contents` with `role: "model"` (not “assistant”) and `parts: [{"text": ...}]` (not a bare string), plus an optional `systemInstruction`.

> 📧 **Analogy**
>
> The data format is the visa for entering Google’s training system. Your content can be perfect, but with the wrong stamp (`role: assistant` instead of `model`, or a string instead of `parts`) you are turned away at the gate - the job fails validation. *Breaks down when:* you assume “JSON is JSON” - the OpenAI messages format and the Gemini contents format are different visas for different countries.

You take the OpenAI-format JSONL from Lesson 9.2 (messages / role: assistant / string content) and submit it to a Vertex Gemini tuning job. What happens?

- `to_gemini` wraps each (user, model) pair in the Gemini schema: `systemInstruction` + `contents` with `role: model` and `parts: [{{text}}]`.
- It serializes to JSONL - one example per line, exactly what the tuning job reads from GCS.
- The printed contrast is the whole gotcha: “assistant”+string (OpenAI) vs “model”+parts (Gemini).

**📝 `01_format.py`** - *Format*

In [ ]:
# GEMINI TUNING DATA FORMAT - role "model" (not "assistant"), parts:[{text:...}].
import json

def to_gemini(system, pairs):
    out = []
    for user, model in pairs:
        out.append({"systemInstruction": {"role": "system", "parts": [{"text": system}]},
                    "contents": [{"role": "user",  "parts": [{"text": user}]},
                                 {"role": "model", "parts": [{"text": model}]}]})
    return out

pairs = [("What is the refund policy?", "Full refund within 7 days, 50% from day 8-30, none after 30."),
         ("How much is the GenAI course?", "14,999 rupees with lifetime access.")]
data = to_gemini("You are a Netsetos support assistant.", pairs)
jsonl = "\n".join(json.dumps(x, ensure_ascii=False) for x in data)   # one example per line
print(f"  {len(data)} examples -> Gemini tuning JSONL. First line (truncated):")
print("   ", jsonl.splitlines()[0][:88], "...")
print("  Gemini vs OpenAI format:")
print("    OpenAI: 'messages', role 'assistant', content is a string")
print("    Gemini: 'contents', role 'model',     parts is [{'text': ...}]  (+ optional systemInstruction)")

# Output:
#   2 examples -> Gemini tuning JSONL. First line (truncated):
#     {"systemInstruction": {"role": "system", "parts": [{"text": "You are a Netsetos support  ...
#   Gemini vs OpenAI format:
#     OpenAI: 'messages', role 'assistant', content is a string
#     Gemini: 'contents', role 'model',     parts is [{'text': ...}]  (+ optional systemInstruction)

#### 💡 What just happened

⚡ What just happened? Two examples become Gemini-shaped JSONL. The load-bearing difference: **role “model” and `parts: [{{text}}]`**, not “assistant” and a string. Get it wrong and the job fails at validation - no training, no useful error beyond “invalid format.” The tradeoff of a managed platform in one line: you give up control over the training loop, and in exchange you must speak the platform’s exact dialect. Upload this JSONL to a GCS bucket and you are ready to launch.

- Click each of the six stages (prepare, upload, job, train, evaluate, deploy) to see what happens and what YOU do vs what Google does.
- See how little of the pipeline you actually touch - the ends, not the middle.

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 2: Launch the Tuning Job - the Current SDK

### Launch the Tuning Job - the Current SDK

One client, one call. The SDK you use is the one that will still exist tomorrow.

This is the core of the lesson - and the cold-open’s payoff. You launch a job with the unified `google-genai` client ([github.com/googleapis/python-genai](https://github.com/googleapis/python-genai)): `client.tunings.tune(base_model=..., training_dataset=TuningDataset(gcs_uri=...), config=CreateTuningJobConfig(...))`. That is it - Google provisions the GPUs, runs LoRA, and auto-deploys the result. The old `vertexai.tuning.sft.train` was **removed in June 2026**, so it is not an option; every example here is on the SDK that still imports.

> 🔊 **Analogy**
>
> Launching a tuning job is placing a catering order, not cooking. You send the order form (base model, dataset, epochs) and the kitchen (Vertex) does everything - provisioning, training, plating on an endpoint. *Breaks down when:* you order from a restaurant that closed - calling the removed `vertexai.tuning` API is exactly that, an order to a kitchen that no longer exists.

Your teammate’s tuning script still calls `vertexai.tuning.sft.train(...)`. It worked last year. Run it today (July 2026). What happens?

- `genai.Client(vertexai=True, project=..., location=...)` is the unified client - one object for Gemini API and Vertex.
- `client.tunings.tune` takes `base_model` (renamed from source_model), a `TuningDataset(gcs_uri=...)`, and a `CreateTuningJobConfig` (epoch_count, learning_rate_multiplier, adapter_size, display name).
- `adapter_size` is the LoRA rank as an enum; poll with `client.tunings.get`, then read `job.tuned_model.endpoint`.

**📝 `02_launch.py`** - *google-genai*

In [ ]:
# LAUNCH A GEMINI TUNING JOB - the CURRENT google-genai SDK.
# pip install google-genai   (the old `vertexai.tuning` was REMOVED 2026-06-24)
from google import genai
from google.genai import types

client = genai.Client(vertexai=True, project="your-project", location="us-central1")

job = client.tunings.tune(
    base_model="gemini-2.5-flash",          # 2.0 is EOL; verify the current tunable lineup
    training_dataset=types.TuningDataset(gcs_uri="gs://YOUR_BUCKET/finetune/train.jsonl"),
    config=types.CreateTuningJobConfig(
        epoch_count=3,                        # was `epochs`
        learning_rate_multiplier=1.0,         # 1.0 default; Pro + small data -> 10
        adapter_size=types.AdapterSize.ADAPTER_SIZE_FOUR,   # LoRA rank; 4 is the sweet spot
        tuned_model_display_name="netsetos-support-v1",
    ),
)
print(f"Job: {job.name}  state: {job.state}")   # JOB_STATE_PENDING

# poll until done, then read the auto-deployed endpoint:
# import time
# while job.state not in ("JOB_STATE_SUCCEEDED", "JOB_STATE_FAILED"):
#     time.sleep(60); job = client.tunings.get(name=job.name); print(job.state)
# print("tuned endpoint:", job.tuned_model.endpoint)

# Output: (illustrative - a real job runs on Vertex AI, ~1-4 hours)
# Job: projects/.../locations/us-central1/tuningJobs/1234567890  state: JOB_STATE_PENDING

#### 💡 What just happened

⚡ What just happened? One `client.tunings.tune` call launches everything; Google runs LoRA and auto-deploys. The renames from the old SDK are the migration in a nutshell: `source_model → base_model`, `train_dataset → training_dataset` (a `TuningDataset`), `epochs → epoch_count`. The tradeoff you are accepting: total convenience (no GPUs, no loop) for total dependence on the platform’s API and calendar - which is exactly why using the current SDK, not the removed one, is a real engineering decision.

- Advance the job and watch it move through JOB_STATE_PENDING -> RUNNING -> SUCCEEDED, with a status log.
- See where you poll, and where the auto-deployed endpoint appears at the end.

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 3: Hyperparameters vs Overfitting

### Hyperparameters vs Overfitting

Flash: defaults. Pro + small data: aggressive. And watch the validation loss.

You expose only a few knobs - `learning_rate_multiplier`, `epoch_count`, and `adapter_size` (the LoRA rank) - because Vertex auto-tunes the rest (batch size is not configurable). The single most important, most-missed rule: **Flash models are fine on defaults, but a Pro model on a small dataset needs an aggressive learning rate and many epochs** (Google says LR×10, 20 epochs). And whatever you set, the loss curves tell you if you overshot.

> 🎓 **Analogy**
>
> Hyperparameters are the intensity of tutoring. A quick learner (Flash) does fine with a normal schedule (defaults). A deeper student on a thin syllabus (Pro + small data) needs intense, repeated sessions (high LR, many epochs) to actually absorb it. *Breaks down when:* you over-drill - too many epochs and the student memorizes the practice questions (training loss falls) but bombs the real exam (validation loss rises). That is overfitting.

You’re tuning Gemini 2.5 *Pro* on just 500 examples. What hyperparameters?

- `recommend_hparams` keys on the model family and dataset size.
- Pro + small (<1000) returns the aggressive recipe (LR 10, 20 epochs); Pro-large is moderate; Flash uses defaults.
- adapter_size stays 4 (the sweet spot) in every case - raise it only for genuinely complex patterns.

**📝 `03_hparams.py`** - *Hyperparams*

In [ ]:
# HYPERPARAMETERS - Flash uses defaults; Pro + a small dataset needs aggressive settings.
def recommend_hparams(model, n_examples):
    pro = "pro" in model
    small = n_examples < 1000
    if pro and small:      # Google's specific best-practice for Pro + small text data
        return dict(learning_rate_multiplier=10, epoch_count=20, adapter_size=4)
    if pro:
        return dict(learning_rate_multiplier=1.0, epoch_count=10, adapter_size=4)
    return dict(learning_rate_multiplier=1.0, epoch_count="default (auto)", adapter_size=4)   # Flash

for model, n in [("gemini-2.5-flash", 500), ("gemini-2.5-pro", 500), ("gemini-2.5-pro", 20000)]:
    print(f"  {model:18s} n={n:<6} -> {recommend_hparams(model, n)}")
print("  Rule: Flash -> defaults work; Pro + small dataset -> aggressive (LR=10, epochs=20).")

# Output:
#   gemini-2.5-flash   n=500    -> {'learning_rate_multiplier': 1.0, 'epoch_count': 'default (auto)', 'adapter_size': 4}
#   gemini-2.5-pro     n=500    -> {'learning_rate_multiplier': 10, 'epoch_count': 20, 'adapter_size': 4}
#   gemini-2.5-pro     n=20000  -> {'learning_rate_multiplier': 1.0, 'epoch_count': 10, 'adapter_size': 4}
#   Rule: Flash -> defaults work; Pro + small dataset -> aggressive (LR=10, epochs=20).

#### 💡 What just happened

⚡ What just happened? The recommender encodes the rule the defaults hide: **Flash defaults, Pro-small-data aggressive (LR×10, 20 epochs), adapter_size 4**. The tradeoff is capacity vs overfitting: more epochs and a bigger adapter can capture more, but past a point they just memorize the training set - which is why you always tune *with* a validation set and let checkpoints hand you the best intermediate model. Rising validation loss while training loss falls is the universal “stop / reduce” signal.

- Slide learning rate, epochs, and adapter size and watch the train and validation loss curves respond.
- Push epochs too high and see validation loss turn up while training loss keeps falling - overfitting, live.

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 4: Cost & Quotas - and No Inference Premium

### Cost & Quotas - and No Inference Premium

Training is tokens times epochs. Serving a tuned model costs the same as the base.

Vertex tuning is cheap because it is LoRA, not full fine-tuning. **Training cost = dataset tokens × epoch_count × a per-million-token rate** (a typical 1,000-example run is a few dollars). The part that surprises people: **there is no inference premium** - a tuned Gemini costs exactly the same per token as the base model, because the adapter rides the shared base endpoint (no dedicated hosting fee). Contrast that with providers that charge a markup to serve a fine-tuned model.

> 💰 **Analogy**
>
> Training cost is the one-time tuition; inference cost is the ongoing salary. On Vertex the tuition is small (LoRA is cheap) and, crucially, the tutored employee is paid the *same salary* as an untutored one - no premium for the extra skill. *Breaks down when:* you forget the tuition scales with epochs - doubling epochs doubles training cost even though inference stays flat.

You fine-tune Gemini Flash. Does serving the tuned model cost MORE per token than the base Flash model?

- `training_cost` is simply `tokens × epochs × rate` - the whole tuning-cost model.
- The three rows show how cost scales with dataset size and epochs.
- The inference line makes the key point concrete: the tuned Flash monthly bill equals the base Flash bill - no premium.

**📝 `04_cost.py`** - *Cost*

In [ ]:
# COST - training is tokens x epochs x rate; inference has NO tuning premium.
# illustrative Jun-2026 rates - verify current pricing.
TRAIN_PER_MTOK = 3.0            # ~ Flash training $/1M training tokens
INFER = {"flash": (0.30, 2.50), "flash-lite": (0.10, 0.40)}   # $/1M in, out (tuned = SAME as base)

def training_cost(dataset_tokens, epochs):
    return dataset_tokens * epochs / 1e6 * TRAIN_PER_MTOK

for toks, ep in [(100000, 4), (1000000, 5), (10000000, 3)]:
    print(f"  {toks:>10,} tokens x {ep} epochs -> ~${training_cost(toks, ep):,.2f} to train")

tin, tout, qpm = 500, 200, 50000
base = (tin*INFER["flash"][0] + tout*INFER["flash"][1]) / 1e6 * qpm
print(f"  Inference (tuned Flash): ${base:,.2f}/mo at {qpm:,} queries -> SAME as base model ($0 tuning premium)")
print("  Contrast: OpenAI charges a fine-tuned-inference markup; Vertex does not (adapter shares the base endpoint).")

# Output:
#      100,000 tokens x 4 epochs -> ~$1.20 to train
#    1,000,000 tokens x 5 epochs -> ~$15.00 to train
#   10,000,000 tokens x 3 epochs -> ~$90.00 to train
#   Inference (tuned Flash): $32.50/mo at 50,000 queries -> SAME as base model ($0 tuning premium)
#   Contrast: OpenAI charges a fine-tuned-inference markup; Vertex does not (adapter shares the base endpoint).

#### 💡 What just happened

⚡ What just happened? Training a 1M-token set for 5 epochs is on the order of a few dollars; serving the tuned model costs the **same** as the base. The tradeoff Vertex optimizes: you pay a small, one-time, epoch-scaled training cost and then nothing extra forever - which is exactly why managed LoRA tuning is so cheap to run in production. (All prices are illustrative Jun-2026 rates; pre-estimate with the token-count API before you launch, and remember quotas: ~10M examples, ~131K tokens/example, one concurrent job by default - in practice a second concurrent job is rejected until the first finishes or you raise the quota, and any single example over the token limit fails the whole job, so validate lengths before you launch.)

- Slide dataset tokens, epochs, and the model tier and watch training cost move.
- Compare tuned-vs-base inference cost and the OpenAI markup - Vertex charges no tuning premium.

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 5: Use & Evaluate the Tuned Model

### Use & Evaluate the Tuned Model

Call the endpoint like any model. Then prove it is actually better than the base.

When the job succeeds, Vertex auto-deploys the adapter and hands you an endpoint. You call it with the same `client.models.generate_content` you use for any Gemini model - just pass the tuned endpoint as the model. Then comes the step most teams skip and shouldn’t: **evaluate base vs tuned on a held-out set**. Vertex’s **Gen AI Eval Service** automates it with both automatic overlap metrics (BLEU/ROUGE - how closely the output matches a reference answer; higher = closer) and LLM-judge rubrics (GENERAL_QUALITY, INSTRUCTION_FOLLOWING).

> 📊 **Analogy**
>
> Evaluation is the post-tutoring exam. You would not tell a parent “the tutoring worked” without a test the student never saw - and you compare the score to before tutoring. *Breaks down when:* you grade on the practice questions - testing on training data always looks great and tells you nothing about the real world. Always hold data out.

During tuning, your training loss keeps dropping but validation loss starts rising after a few epochs. What is happening?

- The tuned model is just another `model=` argument to `client.models.generate_content` - you pass `job.tuned_model.endpoint`.
- The loop runs base and tuned on the same queries so you can eyeball the difference.
- The commented block shows the managed path: `client.evals.evaluate(...)` with rubric metrics for a real base-vs-tuned scorecard.

**📝 `05_evaluate.py`** - *google-genai*

In [ ]:
# USE + EVALUATE THE TUNED MODEL - same client, the tuned endpoint.
from google import genai
client = genai.Client(vertexai=True, project="your-project", location="us-central1")

TUNED = "projects/.../locations/us-central1/endpoints/YOUR_ENDPOINT"   # job.tuned_model.endpoint
tests = ["What is the refund policy?", "Do you offer EMI options?"]

for q in tests:
    base  = client.models.generate_content(model="gemini-2.5-flash", contents=q).text
    tuned = client.models.generate_content(model=TUNED, contents=q).text
    print(f"Q: {q}\n  base : {base[:70]}\n  tuned: {tuned[:70]}")

# Managed base-vs-tuned scoring with the Gen AI Eval Service (GA):
# result = client.evals.evaluate(dataset=eval_ds,
#             metrics=["GENERAL_QUALITY", "INSTRUCTION_FOLLOWING"])   # + BLEU / ROUGE
# print(result.summary_metrics)

# Output: (illustrative - requires a deployed tuned endpoint)
# Q: What is the refund policy?
#   base : Refund policies vary by provider; please check the website ...
#   tuned: Full refund within 7 days, 50% from day 8-30, none after 30 ...

#### 💡 What just happened

⚡ What just happened? The tuned endpoint is called exactly like the base model - same client, different `model=`. The discipline that matters: **always compare base vs tuned on held-out data**, and read the loss curves (rising validation loss = overfitting, so reduce epochs/adapter or grab a checkpoint). The Gen AI Eval Service turns that comparison into a repeatable scorecard. The tradeoff of skipping it: you ship a tuned model with no evidence it beats a plain prompt - the exact mistake Lesson 9.1 warned about.

- Toggle test examples and per-metric scores and watch the base-vs-tuned winner update.
- See why a tuned model that is not clearly better means you should have stayed with prompting or RAG.

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 6: SFT / DPO / Continuous / Gemma - Managed vs Open

### SFT / DPO / Continuous / Gemma - Managed vs Open

Four ways to tune on Vertex, and the one decision that splits them: do you need the weights?

Vertex offers more than one tuning path. **SFT** (supervised) teaches behavior and format; **DPO** (Direct Preference Optimization) teaches preferences (tone, helpfulness) from chosen/rejected pairs - same prompt, a warm reply marked *chosen* over a curt one marked *rejected*; **continuous** tuning keeps improving an already-tuned model; **checkpoints** let you pick the best intermediate state. But the decision that overrides all of them is **Gemini (managed) vs Gemma (open weights)** - and it turns on a single question: do you need to download the trained weights?

> 🏢 **Analogy**
>
> Gemini tuning is renting a fully-serviced apartment: everything works, but you cannot take the furniture when you leave. Gemma is buying a house: you own everything and can move it anywhere - but you fix the plumbing yourself. *Breaks down when:* you conflate “more control” with “better” - most teams are better served renting (managed Gemini); only a real need for the weights (on-prem, multi-cloud, sovereignty) justifies buying (Gemma).

You must fine-tune, but you also need downloadable model weights to deploy on your own on-prem GPUs. Which path?

- `pick_platform` keys on the three things that force open weights: downloadable weights, on-prem/multi-cloud, or data sovereignty.
- Any of those routes to **Gemma** (Model Garden, self-managed); otherwise managed **Gemini** wins on zero ops.
- The reason strings name the tradeoff each way - convenience vs control.

**📝 `06_decide.py`** - *Decision*

In [ ]:
# GEMINI (managed) vs GEMMA (open weights) - the lock-in decision.
def pick_platform(need_downloadable_weights, need_on_prem_or_multicloud, want_zero_gpu_ops):
    if need_downloadable_weights or need_on_prem_or_multicloud:
        return ("Gemma (Model Garden)",
                "open weights: full LoRA/QLoRA, download and deploy anywhere - but you manage the GPUs/serving.")
    if want_zero_gpu_ops:
        return ("Gemini SFT/DPO (managed)",
                "upload JSONL, click train, auto-deployed endpoint - but weights stay on Google's shared endpoint.")
    return ("Gemini SFT/DPO (managed)", "default: the zero-ops path unless you specifically need the weights.")

for label, args in [("Enterprise, on-prem required", (True, True, False)),
                    ("Startup, no GPU team", (False, False, True)),
                    ("Multi-cloud, data sovereignty", (True, True, False))]:
    plat, why = pick_platform(*args)
    print(f"  {label:32s} -> {plat}")
    print(f"      {why}")

# Output:
#   Enterprise, on-prem required     -> Gemma (Model Garden)
#       open weights: full LoRA/QLoRA, download and deploy anywhere - but you manage the GPUs/serving.
#   Startup, no GPU team             -> Gemini SFT/DPO (managed)
#       upload JSONL, click train, auto-deployed endpoint - but weights stay on Google's shared endpoint.
#   Multi-cloud, data sovereignty    -> Gemma (Model Garden)
#       open weights: full LoRA/QLoRA, download and deploy anywhere - but you manage the GPUs/serving.

| Method | Models (Jul 2026) | Best for |
|---|---|---|
| SFT | Gemini 2.5 Pro / Flash / Flash-Lite | Task adaptation, format, domain |
| DPO (preference) | Gemini 2.5 Pro / Flash / Flash-Lite | Tone, style, helpfulness |
| Continuous | Previously tuned models | Iterative improvement |
| Checkpoints | All tunable models | Best intermediate state |
| Gemma (Model Garden) | Gemma open weights (2B–27B) | Downloadable weights, full control |

> 📦 **Info**
>
> ⚖️ DPO at a glance: chosen vs rejected**DPO (Direct Preference Optimization)** does not learn from one ideal answer - it learns from PAIRS where you mark one reply better than another. Same prompt, two completions: a warm, complete reply (`chosen`) and a curt or vague one (`rejected`). One preference example is shaped like `{"contents": [ ...the user turn... ], "chosen": {"role": "model", "parts": [{"text": "..."}]}, "rejected": {"role": "model", "parts": [{"text": "..."}]}}`. You launch it with the same `client.tunings.tune(...)` call on a preference dataset - and Google’s recommended workflow is **SFT first** (teach the format), **then DPO** (refine the tone). This is the data shape the DPO exercise below asks you to build.

#### 💡 What just happened

⚡ What just happened?**SFT + DPO** cover most cases - Google suggests SFT first, then DPO on preference pairs. **Continuous** tuning avoids restarting from scratch (you pass your already-tuned model back in as the base); **checkpoints** hand you the best intermediate model (Vertex saves intermediate states and lets you pick the one with the lowest validation loss). But the decision that dominates is managed-vs-open: **Gemini is zero-ops but locked to Google’s endpoint; Gemma gives you the weights but you run the GPUs.** (Model names move fast - Gemini 2.5 tuning is current now but retires around Oct 2026, with 3.x support rolling out, so verify the tunable lineup in the console.)

- Toggle your requirements (downloadable weights, on-prem, multi-cloud, data sovereignty, zero GPU ops) and watch the recommendation flip between Gemini and Gemma.
- See that only a real need for the WEIGHTS forces the open path - everything else favors managed Gemini.

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 7: Deployment & Constraints

### Deployment & Constraints

Auto-deploy is convenient - the constraints are what decide if it fits.

When tuning succeeds, Vertex does the deployment for you: the adapter is registered in Model Registry and served on a **shared public endpoint** - no manual deploy step. Convenient, but the constraints are the real content. There are **no dedicated/private endpoints** for tuned Gemini, tuning is **not a Covered Service (no SLA)**, and JSON-mode / controlled generation can *degrade* a tuned model. Know these before you promise an enterprise an uptime guarantee.

> 🎪 **Analogy**
>
> The auto-deploy is a shared stage: your act goes on immediately, but it is a public theater - no private room, no guaranteed slot, and some props (JSON mode) can throw off your tuned performance. *Breaks down when:* a client needs a private room with an SLA - managed Gemini tuning cannot give it, so that is a Gemma-self-host signal.

A bank requires a private, dedicated endpoint with an uptime SLA for its tuned support model. Can managed Gemini tuning provide that?

- `deploy_check` maps three requirements (dedicated endpoint, SLA, JSON mode) to Vertex’s tuning constraints.
- Needing a dedicated endpoint or an SLA returns a BLOCK - those are not available for tuned Gemini.
- JSON mode returns a WARN, not a block - it can degrade quality, so test it.

**📝 `07_deploy.py`** - *Constraints*

In [ ]:
# DEPLOYMENT CONSTRAINTS - what managed tuning can and cannot do.
def deploy_check(need_dedicated_endpoint, need_sla, using_json_mode):
    flags = []
    if need_dedicated_endpoint:
        flags.append("BLOCK: tuned Gemini serves ONLY on a shared public endpoint (no dedicated/private).")
    if need_sla:
        flags.append("BLOCK: tuning is not a Covered Service - no uptime SLA.")
    if using_json_mode:
        flags.append("WARN: controlled generation (JSON mode) can degrade a tuned model - test it.")
    return flags or ["OK: within Vertex managed-tuning constraints - auto-deployed, ready to call."]

for label, args in [("Public chatbot", (False, False, False)),
                    ("Bank needing SLA + private endpoint", (True, True, False)),
                    ("Structured-output extractor", (False, False, True))]:
    print(f"  {label}:")
    for f in deploy_check(*args):
        print("    ", f)

# Output:
#   Public chatbot:
#      OK: within Vertex managed-tuning constraints - auto-deployed, ready to call.
#   Bank needing SLA + private endpoint:
#      BLOCK: tuned Gemini serves ONLY on a shared public endpoint (no dedicated/private).
#      BLOCK: tuning is not a Covered Service - no uptime SLA.
#   Structured-output extractor:
#      WARN: controlled generation (JSON mode) can degrade a tuned model - test it.

#### 💡 What just happened

⚡ What just happened? Auto-deploy removes the deployment work but hands you three hard constraints: **shared endpoint only, no SLA, and JSON-mode can hurt a tuned model.** The tradeoff is fully-managed convenience vs enterprise control - a public chatbot is fine on the shared endpoint, but a regulated, SLA-bound workload is a signal to move to Gemma self-hosted. Knowing the constraints up front is what keeps you from promising something the platform cannot deliver.

> 💡 **Info**
>
> 🔄 The lifecycle nobody plans for: base-model retirementYour tuned adapter is pinned to its base model - and base models retire (Gemini 2.5 tuning retires around Oct 2026, exactly as 2.0 and 1.5 did before it). When the base retires, your tuned model stops serving, and you **cannot migrate a tuned model across base versions**: you must run a fresh tuning job on the successor base and re-evaluate. So put the base’s retirement date on your calendar the day you tune - the same discipline as the SDK-removal date in the cold-open. On a managed platform, currency is a maintenance schedule, not a one-time fix.

---

## 🎯 Concept 8: India - Regions and DPDP

### India - Regions and DPDP

Inference is in Mumbai. Tuning likely is not. That gap is the compliance story.

The India challenge is a **region gap**. Inference runs from **asia-south1 (Mumbai)** for all Gemini tiers, but **tuning is confirmed in us-central1 / europe-west4** (asia-south1 support is emerging). So the practical pattern is **tune in us-central1, serve from asia-south1** - which means training data crosses borders. Under DPDP that requires deliberate safeguards, and for the strictest cases it points back to Gemma-in-India.

> 🗺️ **Analogy**
>
> It is like sending documents abroad to be processed and getting the results back home. The processing (tuning) happens overseas (us-central1); the service desk (inference) is local (Mumbai). *Breaks down when:* the documents contain personal data and the law restricts sending it abroad - then you either wrap the transfer in safeguards (CMEK, VPC-SC, Zero Data Retention, a DPIA) or keep everything in-country with Gemma on IndiaAI GPUs.

You’re building a support bot for an Indian bank; the training data contains customer PII. Can you use Vertex fine-tuning?

- `india_plan` always starts with the region pattern: tune in us-central1, serve from asia-south1.
- PII adds the DPDP safeguards (Presidio removal first, then CMEK / VPC-SC / ZDR / DPIA).
- RBI-regulated adds the sovereignty escape hatch: Gemma on IndiaAI GPUs, training entirely in India.

**📝 `08_india.py`** - *India*

In [ ]:
# INDIA - the region gap: inference in Mumbai, tuning crosses borders. Plan for DPDP.
def india_plan(has_pii, rbi_regulated):
    plan = ["tune in us-central1 (or europe-west4); serve inference from asia-south1 (Mumbai) for low latency"]
    if has_pii:
        plan.append("remove PII with Presidio BEFORE upload (lesson 9.2)")
        plan += ["enable CMEK", "enable VPC Service Controls", "enable Zero Data Retention", "run a DPDP DPIA"]
    if rbi_regulated:
        plan.append("for full data sovereignty, prefer Gemma on IndiaAI GPUs - training never leaves India")
    return plan

print("  Indian bank support bot (PII, RBI-regulated):")
for i, step in enumerate(india_plan(has_pii=True, rbi_regulated=True), 1):
    print(f"    {i}. {step}")
print("  Region gap: inference is in Mumbai, but tuning crosses borders to us-central1 - plan for DPDP.")

# Output:
#   Indian bank support bot (PII, RBI-regulated):
#     1. tune in us-central1 (or europe-west4); serve inference from asia-south1 (Mumbai) for low latency
#     2. remove PII with Presidio BEFORE upload (lesson 9.2)
#     3. enable CMEK
#     4. enable VPC Service Controls
#     5. enable Zero Data Retention
#     6. run a DPDP DPIA
#     7. for full data sovereignty, prefer Gemma on IndiaAI GPUs - training never leaves India
#   Region gap: inference is in Mumbai, but tuning crosses borders to us-central1 - plan for DPDP.

> 📦 **Info**
>
> 🇮🇳 India: regions + DPDP (Jul 2026)
> - **Inference:** asia-south1 (Mumbai) - Gemini Flash / Flash-Lite / Pro, low latency for Indian users.
> - **Tuning:** confirmed in us-central1 and europe-west4; asia-south1 is emerging - verify in the console before assuming.
> - **Pattern:** tune once in us-central1, serve continuously from asia-south1.
> - **DPDP Act 2023:** consent-centric, penalties up to ₹250 crore, full enforcement expected ~May 2027. Cross-border training data needs **CMEK** (your own encryption keys), **VPC Service Controls** (a network boundary that blocks data exfiltration), **Zero Data Retention** (Google stores none of your prompts or outputs), and a **DPIA** (a written data-protection risk assessment), plus audit logging.
> - **Full sovereignty:** for RBI-regulated data, use Gemma + IndiaAI GPUs - training never leaves India.

#### 💡 What just happened

⚡ What just happened? India’s tuning story is the region gap: **Mumbai for inference, us-central1 for tuning**, so training data crosses borders. The tradeoff that decides your architecture is convenience vs sovereignty - managed Vertex with DPDP safeguards (CMEK / VPC-SC / ZDR / DPIA) is workable for most, but the strictest, RBI-regulated cases keep everything in India with Gemma on subsidized GPUs. Either way, strip PII before anything leaves your control (Lesson 9.2).

## Putting It Together

> ✅ **Info**
>
> 🧠 The managed-tuning pipeline in one path
> - **Format + upload.** Gemini `contents` / `role: model` / `parts` JSONL to GCS (from Lesson 9.2).
> - **Launch on the current SDK.** `client.tunings.tune(...)` on `google-genai` - never the removed `vertexai.tuning`.
> - **Set hyperparameters.** Flash defaults; Pro-small-data aggressive; adapter_size 4; watch validation loss.
> - **Cost + use + evaluate.** Cheap LoRA training, no inference premium; call the endpoint; prove base-vs-tuned on held-out data.
> - **Choose your lock-in.** Managed Gemini (zero ops, no weights) vs Gemma (weights, self-managed); handle constraints + India/DPDP.
> The through-line: **managed tuning trades control for convenience - you speak the platform’s exact dialect and live on its calendar, and in return you never touch a GPU.**

> 📦 **Info**
>
> 🔗 Where this goes next
> - In Lesson 9.4 we’ll take the open-weight path this lesson kept pointing at - **Gemma / open-source fine-tuning** (Axolotl / Unsloth / TRL), where you DO get the weights.
> - Lesson 9.5 picks this up mechanically - the **LoRA & QLoRA** adapters Vertex runs for you, done by hand.
> - In Lesson 9.6 we’ll come back to **evaluation, merging, and deployment** beyond the managed defaults.
> - Later, in Module 14 (**LLMOps**), we’ll return to monitoring a tuned model in production.

*Practice exercises are in the companion practice notebook.*

---

## 🎓 Summary

You've completed the practical part of **Fine-Tuningwith Vertex AI**.

### Next steps:
1. Re-run cells with different parameters to build intuition
2. Try the practice exercises (see `practice-lab/practice-lab-9_3.ipynb` if available)
3. Review the full HTML lesson for the complete narrative

*Generated from `lesson-9.3.html` - regenerate this notebook whenever the source HTML is updated.*
